# Tier 1.1: The problem and the data

ET-Miner finds sets of items that occur together in many rows of a table.
This notebook defines that problem on a table small enough to check by hand,
shows the input format the library expects, and shows how the library turns
that input into the representation it counts on.

**What you will learn**

- what a transaction, an item, an itemset and its support are;
- the input format of `et_miner.apriori` (one list column per row);
- how `src/et_miner/core/matrix.py:build_boolean_matrix` turns the rows into a boolean matrix;
- how a support fraction becomes an integer minimum count;
- the shared sample that tiers 1, 2 and 3 all mine.

**Prerequisites**

- Python and basic Polars.
- The environment from [the docs README](../README.md#3-environment-setup).
- Terms are defined in [concepts.md](../concepts.md).

## 0. Setup

The cell below imports the library and prints the versions this notebook ran with.
`LOGURU_LEVEL` hides the library's debug log lines so the outputs stay readable.

In [1]:
import os
import warnings

os.environ.setdefault("LOGURU_LEVEL", "WARNING")
warnings.filterwarnings("ignore", message="IProgress not found")

import json
from pathlib import Path

import polars as pl

import et_miner
from et_miner import apriori

DATA = Path("../data")
print("et_miner", et_miner.__version__, "| polars", pl.__version__)

et_miner 0.2.0 | polars 1.43.2


## 1. A co-occurrence pattern

A **transaction** (or row) is a set of **items**, for example the products in one shopping basket.
An **itemset** is any set of items.
Its **support** is the fraction of transactions that contain every item of the set.
An itemset is **frequent** when its support reaches a threshold called `min_support`.
The task is to list every frequent itemset and its support.

The next cell builds an 8-row table with 6 items (`a` to `f`) by hand.
Look at the rows: you will count items in them in the sections below.

In [2]:
rows = [
    ["a", "b", "c"],
    ["a", "b"],
    ["a", "c", "d"],
    ["b", "c"],
    ["a", "b", "c", "e"],
    ["a", "b", "d"],
    ["c", "e", "f"],
    ["a", "b", "c", "f"],
]
toy = pl.DataFrame({"items": rows})
toy

items
list[str]
"[""a"", ""b"", ""c""]"
"[""a"", ""b""]"
"[""a"", ""c"", ""d""]"
"[""b"", ""c""]"
"[""a"", ""b"", … ""e""]"
"[""a"", ""b"", ""d""]"
"[""c"", ""e"", ""f""]"
"[""a"", ""b"", … ""f""]"


The table has one column, `items`, and each cell holds a list.
That is the whole input format.

## 2. The input format

`apriori` takes a Polars `DataFrame` or `LazyFrame` with one list column.
The column is called `items` by default; `item_col=` selects another name.
Items can be integers or strings.
The same table is stored in `docs/data/toy_8x6.parquet`, written by `docs/data/make_samples.py`.
The next cell reads it back and checks that it equals the table above.

In [3]:
toy_file = pl.read_parquet(DATA / "toy_8x6.parquet")
print(toy_file.schema)
assert toy_file.equals(toy)
print("file matches the hand-built table")

Schema([('items', List(String))])
file matches the hand-built table


The file holds the same eight rows, typed `list[str]`.

## 3. From a support fraction to a minimum count

The library compares integer counts, not fractions.
It converts `min_support` to a minimum count with
`src/et_miner/core/result.py:_min_count`, which computes `ceil(min_support * n_rows)` exactly
(it reads the float as the decimal it prints as, so `0.07` means exactly 7/100).
The next cell prints the minimum count for a few thresholds on 8 rows.
Look at how coarse the steps are.

In [4]:
from et_miner.core.result import _min_count

for s in [0.1, 0.125, 0.2, 0.25, 0.3, 0.375, 0.5]:
    print(f"min_support={s:<6} -> min_count={_min_count(s, len(toy))}")

min_support=0.1    -> min_count=1
min_support=0.125  -> min_count=1
min_support=0.2    -> min_count=2
min_support=0.25   -> min_count=2
min_support=0.3    -> min_count=3
min_support=0.375  -> min_count=3
min_support=0.5    -> min_count=4


On 8 rows every threshold in (0, 0.125] means "appears at least once", and 0.2 and 0.25 both mean 2.
On small inputs a small fraction therefore keeps every item and every subset of every row.
This notebook series uses `min_support=0.25`, which means a count of at least 2.

## 4. From rows to a boolean matrix

The Polars tier does not count on the lists directly.
`build_boolean_matrix` first counts single items, keeps the frequent ones,
and builds one boolean column per frequent item (one row per transaction).
Column names are `i_0`, `i_1`, ... and `col_to_item` maps them back to item values.
The next cell runs it at `min_support=0.25`.

In [5]:
from et_miner.core.matrix import build_boolean_matrix

matrix, col_to_item, n_rows = build_boolean_matrix(toy.lazy(), 0.25)
print("n_rows =", n_rows)
print("col_to_item =", col_to_item)
matrix

n_rows = 8
col_to_item = {'i_0': 'a', 'i_1': 'b', 'i_2': 'c', 'i_3': 'd', 'i_4': 'e', 'i_5': 'f'}


i_0,i_1,i_2,i_3,i_4,i_5
bool,bool,bool,bool,bool,bool
true,true,true,false,false,false
true,true,false,false,false,false
true,false,true,true,false,false
false,true,true,false,false,false
true,true,true,false,true,false
true,true,false,true,false,false
false,false,true,false,true,true
true,true,true,false,false,true


Every one of the six items is frequent at a count of 2, so the matrix has six columns.
Each cell answers one question: does row *r* contain item *i*.
Support counting is then a column AND followed by a sum.

## 5. Counting one itemset by hand and with the matrix

The next cell counts the itemset `{a, b}` in two ways:
directly on the lists, and as `(col_a & col_b).sum()` on the matrix.
Both numbers must agree.

In [6]:
item_to_col = {v: k for k, v in col_to_item.items()}
by_lists = sum(1 for r in rows if {"a", "b"} <= set(r))
by_matrix = matrix.select((pl.col(item_to_col["a"]) & pl.col(item_to_col["b"])).sum()).item()
print("rows containing {a, b}:", by_lists, "(lists) |", by_matrix, "(matrix)")
assert by_lists == by_matrix

rows containing {a, b}: 5 (lists) | 5 (matrix)


Both methods count the same rows.
The library's `src/et_miner/core/matrix.py:count_support_vectorized` builds this expression for many itemsets at once.

## 6. The full answer on the toy table

The next cell calls the real miner.
`sparse=False` forces the Polars counting path (tier 1); see
[concepts.md, section 6](../concepts.md#6-routes-and-tiers) for why this matters when the Rust extension is installed.
The result has two columns, `itemset` and `support`.

In [7]:
result = apriori(toy, min_support=0.25, sparse=False)
result.with_columns(count=(pl.col("support") * n_rows).round().cast(pl.Int64)).sort(
    pl.col("itemset").list.len(), "itemset"
)

itemset,support,count
list[str],f64,i64
"[""a""]",0.75,6
"[""b""]",0.75,6
"[""c""]",0.75,6
"[""d""]",0.25,2
"[""e""]",0.25,2
…,…,…
"[""a"", ""d""]",0.25,2
"[""b"", ""c""]",0.5,4
"[""c"", ""e""]",0.25,2


There are 13 frequent itemsets: six single items, six pairs and one triple.
You can check each count against the table in section 1; notebook 2 derives them level by level.

## 7. The shared sample

The tutorials use one larger sample across all three tiers: the `smoke` preset of
`src/et_miner/synthetic.py`, saved to `docs/data/smoke.parquet`.
It has Zipf-distributed item popularity and three planted 5-item motifs.
It is the same dataset that `tests/test_tier_equivalence.py` mines.
The next cell loads it and prints its shape and parameters.

In [8]:
smoke = pl.read_parquet(DATA / "smoke.parquet")
meta = json.loads((DATA / "smoke.json").read_text())
print(smoke.shape, smoke.schema)
print({k: meta[k] for k in ["n_rows", "vocab_size", "row_len_mean", "motif_count", "motif_size", "min_support", "min_count"]})
lengths = smoke.select(pl.col("items").list.len().alias("row_length")).describe()
lengths

(60000, 1) Schema([('items', List(Int64))])
{'n_rows': 60000, 'vocab_size': 2000, 'row_len_mean': 9, 'motif_count': 3, 'motif_size': 5, 'min_support': 0.01, 'min_count': 600}


statistic,row_length
str,f64
"""count""",60000.0
"""null_count""",0.0
"""mean""",8.30915
"""std""",2.869904
"""min""",1.0
"""25%""",6.0
"""50%""",8.0
"""75%""",10.0
"""max""",25.0


The sample has 60,000 rows over a vocabulary of 2,000 items, and the tutorials mine it at
`min_support=0.01`, a minimum count of 600.

## Summary and next step

- The input is one list column; each row is a transaction.
- `min_support` becomes an integer count `ceil(min_support * n_rows)`; on small inputs that count is coarse.
- Tier 1 counts on a boolean matrix with one column per frequent item.

Next: [02-apriori-step-by-step](02-apriori-step-by-step.ipynb) runs the algorithm one level at a time.